# Tool Environments

Every tool in `proto_tools` runs inside its own isolated micromamba environment. The first time you call a tool, the library detects your hardware, resolves that tool's dependencies, installs them into a sandbox under `$PROTO_HOME/proto_tool_envs/`, and runs the tool from there. Every call after that reuses the cached environment and skips straight to execution.

The payoff is that tools with mutually incompatible dependencies coexist in a single script. One tool can want PyTorch 2.1 while another wants 2.4, one can pin an old Python and another the latest, and you never have to think about it: you import a tool and call it, and the library keeps each one in its own corner.

## The problem it solves

Computational biology tools are notoriously hard to install side by side. AlphaFold3 wants JAX built against a specific CUDA version; ESM2 expects a particular PyTorch release; BLAST is a system binary that has to be on your PATH; RFdiffusion pins its Python version; and each one likes to keep its weights somewhere different. Getting any two of them into the same environment is a gamble, and getting four of them there is a weekend lost to dependency resolution.

The traditional workaround is one conda environment per tool, activated by hand before every run. Proto Tools removes that chore: you import a tool and call it, and the library builds and manages the right environment underneath. Because no two tools actually share an environment, their conflicting requirements simply never meet.

<img src="assets/tool-environments/architecture.svg" alt="Your script calls into proto_tools, which dispatches each tool into its own isolated environment" style={{width: "100%", maxWidth: "560px", display: "block", margin: "1.5rem auto"}} />

## What's in an environment

Each environment lives at `$PROTO_HOME/proto_tool_envs/{tool_key}_env/` and is a complete, self-contained sandbox. Inside it is an isolated Python interpreter pinned to the version the tool needs, the tool's dependencies resolved as hardware-aware wheels (the right CUDA build for your GPU, or a CPU-only stack on a laptop), and a local cache for the compiled extensions and JIT artifacts the tool produces at runtime. Tools that ship a command-line binary, such as BLAST or MAFFT, have that binary installed into the environment as well, and any variables the tool declares in its `env_vars.txt` (Hugging Face tokens, backend selection, and the like) are applied automatically whenever it runs.

Model weights are the one thing kept outside the environment. They live separately in `$PROTO_HOME/proto_model_cache/{tool_key}/` (overridable with `PROTO_MODEL_CACHE`), so tearing an environment down and rebuilding it never re-downloads a multi-gigabyte checkpoint.

## First call vs. cached call

The first time you call a tool on a given machine, the library does the full setup. It reads the tool's standalone setup scripts, detects your hardware (GPU availability, CUDA version, OS, and architecture), and resolves the dependency set that fits, choosing the right PyTorch or JAX wheel, the required Python version, and any binaries. It then creates the isolated micromamba environment under `$PROTO_HOME/proto_tool_envs/`, installs everything into it, and finally launches the tool and returns the result.

<img src="assets/tool-environments/lifecycle.svg" alt="First call installs the env; subsequent calls reuse the cache" style={{width: "100%", maxWidth: "560px", display: "block", margin: "1.5rem auto"}} />

That cost is paid exactly once. Every later call finds the environment already in place and goes straight to execution, so setup is a one-time price per tool, per machine. It is easiest to see by watching the environment directory as it happens. Before the first call, nothing is there yet, because no tool has needed a sandbox:

In [ ]:
import os

os.environ.setdefault("PROTO_HOME", os.path.expanduser("~/.proto"))
!ls $PROTO_HOME/proto_tool_envs/ 2>/dev/null || echo "(no tool environments yet)"

(no tool environments yet)


Now call ESM2 for the first time. We use the small `esm2_t6_8M_UR50D` checkpoint on CPU so the walkthrough runs anywhere. This first call builds the `esm2_env` environment before it runs the model, which is why it takes noticeably longer than the embedding itself:

In [ ]:
from proto_tools.tools.masked_models.esm2.esm2_embeddings import (
    ESM2EmbeddingsConfig,
    ESM2EmbeddingsInput,
    run_esm2_embeddings,
)

# First call: builds the isolated environment, then runs inference.
output = run_esm2_embeddings(
    ESM2EmbeddingsInput(sequences=["MKTLLILAVVAAALA"]),
    ESM2EmbeddingsConfig(model_checkpoint="esm2_t6_8M_UR50D", device="cpu"),
)
print(output.embeddings[0].shape)

The environment is on disk now, and it stays there for every future call:

In [ ]:
!ls $PROTO_HOME/proto_tool_envs/

esm2_env


A second call finds that environment already built and goes straight to inference, with no install step in between:

In [ ]:
# Second call: reuses esm2_env, no install step.
output = run_esm2_embeddings(
    ESM2EmbeddingsInput(sequences=["GAVLTVLLGGLLLA"]),
    ESM2EmbeddingsConfig(model_checkpoint="esm2_t6_8M_UR50D", device="cpu"),
)

## Hardware awareness

A single shared detection routine works out what hardware is available, including the CUDA version, GPU architecture, operating system, and processor architecture, and feeds that into every tool's setup. The practical effect is that each tool installs the build that actually matches your machine instead of a lowest-common-denominator one. The PyTorch or JAX wheel is chosen for your specific GPU, a tool that runs CPU-only on a laptop and GPU-accelerated on a server adapts to wherever it finds itself, and CUDA-JIT tools know which GCC and nvcc versions to request when they compile. You get the right environment without configuring anything.

## Debugging setup

If a tool environment fails to install, which happens most often on first contact with a new HPC cluster, turn on verbose logging:

```bash
export PROTO_ENV_VERBOSE=1
export PROTO_ENV_LOG_DIR=./env-logs
```

You will then see the actual `micromamba` and `pip` commands the library is running, along with any errors raised by the tool's `setup.sh`. Every setup attempt also writes a log file into `PROTO_ENV_LOG_DIR` for later inspection.

## Go deeper

For the full implementation reference (standalone setup patterns, `env_vars.txt` conventions, GCC/nvcc compatibility matrices, the `to_device()` protocol, pinned-version tools, and how to write a new tool env), see the canonical developer notes in the proto-tools repo:

<Card title="Tool Environments Reference" icon="book" href="https://github.com/proto-bio/proto-tools/blob/main/notes/tool-environments.md">
  Standalone env setup, compute deps, GCC/nvcc, caches, binaries, and the `to_device()` protocol.
</Card>

## Next Steps

<CardGroup cols={2}>
  <Card title="Device Management" icon="microchip" href="/tools/guides/device-management">
    How tools are placed on GPUs, with LRU eviction and CPU offload.
  </Card>
  <Card title="Tool Persistence" icon="box" href="/tools/guides/tool-persistence">
    Keep a model loaded across calls to skip repeated load times.
  </Card>
  <Card title="Parallel Execution" icon="layers" href="/tools/guides/parallel-execution">
    Fan work out across every available GPU.
  </Card>
  <Card title="Cloud Inference" icon="cloud" href="/tools/guides/cloud-inference">
    Dispatch tool runs to remote compute.
  </Card>
</CardGroup>